# Analisis Top Opening — Lichess Standard Rated 2013

Notebook ini membaca hasil ingestion di `data/processed/games_2013-*` (344 file Parquet, 3.386.423 game) dan menjawab:

1. Opening apa yang paling banyak dimainkan sepanjang 2013?
2. ECO code mana yang dominan?
3. Bagaimana tren opening terpopuler per kuartal?
4. Opening mana yang memberi win rate putih tertinggi?

Prasyarat: dataset sudah di-ingest (lihat README). Notebook tidak pernah menyentuh `data/raw/`.

In [1]:
from pathlib import Path

import polars as pl

# Cari root proyek dari lokasi notebook
ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

PROCESSED = ROOT / "data" / "processed"
files = sorted(PROCESSED.glob("games_2013-*/part-*.parquet"))
assert files, f"Tidak ada file Parquet di {PROCESSED} — jalankan ingestion dulu"

# Kolom 'month' berasal dari nama folder games_2013-MM
frames = [
    pl.scan_parquet(f)
    .select(["eco", "opening", "result", "total_plies"])
    .with_columns(pl.lit(f.parent.name.replace("games_", "")).alias("month"))
    for f in files
]
df = pl.concat(frames, how="vertical").collect()
print(f"Loaded {df.height:,} games dari {len(files)} file Parquet, {df['month'].n_unique()} bulan")

Loaded 3,386,423 games dari 344 file Parquet, 12 bulan


## 1. Top 15 opening sepanjang 2013

In [2]:
top_openings = (
    df.group_by("opening")
    .agg(
        pl.len().alias("games"),
        (pl.col("result") == "1-0").sum().alias("white_win"),
        (pl.col("result") == "0-1").sum().alias("black_win"),
        (pl.col("result") == "1/2-1/2").sum().alias("draw"),
        pl.col("total_plies").mean().round(1).alias("avg_plies"),
    )
    .sort("games", descending=True)
    .head(15)
    .with_columns(
        (pl.col("white_win") / pl.col("games") * 100).round(1).alias("white_win_pct"),
        (pl.col("draw") / pl.col("games") * 100).round(1).alias("draw_pct"),
    )
    .select(["opening", "games", "white_win_pct", "draw_pct", "avg_plies"])
)
top_openings

opening,games,white_win_pct,draw_pct,avg_plies
str,u32,f64,f64,f64
"""Van't Kruijs Opening""",86329,44.1,2.6,65.3
"""Horwitz Defense""",65485,54.4,2.9,68.9
"""Scandinavian Defense: Mieses-K…",64501,57.2,3.3,68.6
"""Modern Defense""",60795,51.2,3.0,69.0
"""Owen Defense""",59461,51.5,2.7,65.9
…,…,…,…,…
"""Bishop's Opening""",42867,55.6,3.2,66.4
"""Sicilian Defense""",42422,42.2,2.9,66.2
"""Scandinavian Defense""",42119,48.2,2.8,63.0


## 2. Top 10 ECO code

ECO mengelompokkan opening jadi keluarga besar — berguna untuk melihat jenis permainan yang dominan.

In [3]:
top_eco = (
    df.group_by("eco")
    .agg(pl.len().alias("games"))
    .sort("games", descending=True)
    .head(10)
    .with_columns((pl.col("games") / df.height * 100).round(1).alias("pct"))
)
top_eco

eco,games,pct
str,u32,f64
"""A00""",240633,7.1
"""C00""",188533,5.6
"""A40""",171838,5.1
"""B01""",146455,4.3
"""D00""",145486,4.3
"""B00""",139863,4.1
"""C41""",133200,3.9
"""C20""",132636,3.9
"""B20""",92790,2.7


## 3. Top 3 opening per kuartal (tren)

In [4]:
df_quarters = df.with_columns(
    pl.when(pl.col("month").str.slice(5, 2).cast(pl.Int32) <= 3).then(pl.lit("Q1"))
    .when(pl.col("month").str.slice(5, 2).cast(pl.Int32) <= 6).then(pl.lit("Q2"))
    .when(pl.col("month").str.slice(5, 2).cast(pl.Int32) <= 9).then(pl.lit("Q3"))
    .otherwise(pl.lit("Q4"))
    .alias("quarter")
)

for q in ["Q1", "Q2", "Q3", "Q4"]:
    top3 = (
        df_quarters.filter(pl.col("quarter") == q)
        .group_by("opening")
        .agg(pl.len().alias("games"))
        .sort("games", descending=True)
        .head(3)
    )
    print(f"{q}: " + "; ".join(f"{r['opening']} ({r['games']:,})" for r in top3.iter_rows(named=True)))

Q1: Van't Kruijs Opening (11,998); Owen Defense (9,035); Scandinavian Defense: Mieses-Kotroc Variation (7,752)
Q2: Van't Kruijs Opening (14,438); Horwitz Defense (11,930); Scandinavian Defense: Mieses-Kotroc Variation (10,171)
Q3: Van't Kruijs Opening (23,784); Scandinavian Defense: Mieses-Kotroc Variation (18,475); Horwitz Defense (17,762)
Q4: Van't Kruijs Opening (36,109); Horwitz Defense (28,600); Scandinavian Defense: Mieses-Kotroc Variation (28,103)


## 4. Win rate putih tertinggi (min 10.000 game)

Ambang 10.000 game mencegah opening langka dengan sampel kecil mendominasi peringkat.

In [5]:
best_white = (
    df.group_by("opening")
    .agg(
        pl.len().alias("games"),
        (pl.col("result") == "1-0").sum().alias("white_win"),
    )
    .filter(pl.col("games") >= 10_000)
    .with_columns((pl.col("white_win") / pl.col("games") * 100).round(1).alias("white_win_pct"))
    .sort("white_win_pct", descending=True)
    .head(5)
    .select(["opening", "games", "white_win_pct"])
)
best_white

opening,games,white_win_pct
str,u32,f64
"""King's Gambit""",10593,63.6
"""Philidor Defense""",15977,62.6
"""Queen's Gambit Accepted""",12398,62.2
"""Queen's Gambit Accepted: Old V…",10903,61.6
"""King's Pawn Game: Busch-Gass G…",10725,59.5


## Interpretasi

Daftar ini mencerminkan **demografi Lichess 2013** (rata-rata Elo ~1.600, komunitas sedang tumbuh), bukan kekuatan teoretis opening:

- **Van't Kruijs (1.e3)** dan **Horwitz Defense** mendominasi — opening yang jarang dibahas buku teori, khas coba-coba pemula.
- **Bowdler Attack (2.Bc4 vs Sicilian)** masuk top 7 tapi win rate putihnya terburuk di top 15 (42,1%) — teori klasik benar: langkah pasif dihukum.
- **King's Gambit 63,6%** — gambit romantis yang ditinggalkan grandmaster bekerja sangat baik di level amatir karena jarang dijawab benar.
- Draw hanya ~3% — mayoritas game blitz/bullet, jauh di bawah catur klasik.

Kesimpulan: di level rating ini, keakraban dengan pola taktik sederhana lebih menentukan hasil daripada keunggulan teori.